In [9]:
import pandas as pd

In [10]:
# Load the CSV file in a DataFrame.
df = pd.read_csv('../../data/raw/stocks/stocks.csv')

# Cleaning

In [11]:
# Extract the ticker (only keep the first if multiple values are present, usually due to multiple cotations)
df['ticker'] = df['ticker'].str.extract(r'^([^(/]+)', expand=False).str.strip()
df.head(30)

,company,ticker,Layer
0,MP Materials,MP,Layer 1
1,USA Rare Earth,USAR,Layer 1
2,Lynas Rare Earths,LYC,Layer 1
3,Energy Fuels,UUUU,Layer 1
4,Albemarle,ALB,Layer 1
5,SQM,SQM,Layer 1
6,Ganfeng Lithium,002460,Layer 1
7,Toray Industries,3402,Layer 2
8,Hexcel,HXL,Layer 2
9,Mitsubishi Chemical Group,4188,Layer 2


Remove non publicly traded companies.

In [12]:
remove_private = (df["ticker"].isin(["Private", "Non-Profit"]) == False)
df_private = df[remove_private].reset_index(drop=True)
df_private["ticker"] = df_private["ticker"].str.strip().str.upper()
df_private

,company,ticker,Layer
0,MP Materials,MP,Layer 1
1,USA Rare Earth,USAR,Layer 1
2,Lynas Rare Earths,LYC,Layer 1
3,Energy Fuels,UUUU,Layer 1
4,Albemarle,ALB,Layer 1
...,...,...,...
100,PTC,PTC,Layer 13
101,ServiceNow,NOW,Layer 13
102,TeamViewer,TMV,Layer 13
103,Trimble,TRMB,Layer 13


Merge the companies sharing the same ticker

In [13]:
df_private["layer_ids"] = df_private["Layer"].str.findall(r"(\d+)").apply(lambda ids: {int(i) for i in ids})
# Layer ids are merged and sorted numerically (10 before 3), not lexicographically.
df_merged = df_private.groupby("ticker", as_index=False).agg(
    layer_ids=("layer_ids", lambda ids: sorted({i for lst in ids for i in lst})),
    company=("company", lambda names: " / ".join(sorted(set(names)))),
)
df_merged["layer"] = df_merged["layer_ids"].apply(lambda ids: ", ".join(f"Layer {i}" for i in ids))
df_merged = df_merged.drop(columns="layer_ids").reset_index(drop=True)
df_merged

,ticker,company,layer
0,000333,KUKA,Layer 11
1,002050,Sanhua Intelligent Controls,Layer 9
2,002230,iFlytek,Layer 3
3,002460,Ganfeng Lithium,Layer 1
4,002594,BYD,Layer 6
...,...,...,...
97,UUUU,Energy Fuels,Layer 1
98,VPG,Vishay Precision,Layer 7
99,WMT,Walmart,Layer 12
100,WOLF,Wolfspeed,Layer 6


Removing company names as they can be found later with the yfinance library via the ticker.

In [14]:
df_cleaned = df_merged.drop(["company"], axis=1)
df_cleaned

,ticker,layer
0,000333,Layer 11
1,002050,Layer 9
2,002230,Layer 3
3,002460,Layer 1
4,002594,Layer 6
...,...,...
97,UUUU,Layer 1
98,VPG,Layer 7
99,WMT,Layer 12
100,WOLF,Layer 6


Create a simple dataset with only the tickers.

In [15]:
df_tickers = df_cleaned.drop(["layer"], axis=1)
df_tickers

,ticker
0,000333
1,002050
2,002230
3,002460
4,002594
...,...
97,UUUU
98,VPG
99,WMT
100,WOLF


# Export

In [16]:
df_cleaned.to_csv('../../data/processed/stocks/stocks_cleaned.csv', index=False)
df_tickers.to_csv('../../data/processed/stocks/tickers.csv', index=False)